In [13]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

In [14]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit_aer import AerSimulator
import math
import numpy as np

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.
simulator = AerSimulator()

In [15]:
def quantum_random_bits(n: int) -> list[int]:
    """
    Generate n truly random bits using quantum measurement of |+> states.
    """
    qc = QuantumCircuit(n, n)
    for i in range(n):
        qc.h(i)
    qc.measure(range(n), range(n))
    job = simulator.run(qc, shots=1, memory=True)
    bitstring = job.result().get_memory()[0]
    return [int(b) for b in reversed(bitstring)]

print("Quantum RNG ready. Sample:", quantum_random_bits(8))

Quantum RNG ready. Sample: [0, 1, 1, 0, 1, 0, 1, 1]


In [16]:
def alice_prepare_qubit(bit: int, basis: int) -> QuantumCircuit:
    """
    ALICE: Prepare a qubit encoding `bit` in `basis`.
      basis=0 (Z-basis): |0> or |1>
      basis=1 (X-basis): |+> or |->
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc


def measure_qubit(circuit: QuantumCircuit, basis: int) -> tuple[int, QuantumCircuit]:
    """
    Measure `circuit` in `basis` (0=Z, 1=X).
    Returns (measured_bit, new_circuit_prepared_from_result).
    The new circuit can be forwarded to the next party.
    """
    qc = circuit.copy()
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    job    = simulator.run(qc, shots=1, memory=True)
    result = int(job.result().get_memory()[0])
    # Re-prepare a fresh qubit from the measurement result (for forwarding)
    new_qc = alice_prepare_qubit(result, basis)
    return result, new_qc


def bob_measure(circuit: QuantumCircuit, basis: int) -> int:
    """
    BOB: Measure `circuit` in `basis` (0=Z, 1=X). Returns 0 or 1.
    """
    result, _ = measure_qubit(circuit, basis)
    return result


def sift_key(alice_bases, bob_bases, alice_bits, bob_results):
    """Return sifted keys for Alice and Bob where bases matched."""
    matching = [i for i in range(len(alice_bases)) if alice_bases[i] == bob_bases[i]]
    return (
        [alice_bits[i]    for i in matching],
        [bob_results[i]   for i in matching],
        matching
    )


def estimate_qber(alice_key, bob_key, sample_fraction=0.25):
    """Sample a fraction of the sifted key and compute QBER."""
    n           = len(alice_key)
    sample_size = max(1, int(n * sample_fraction))
    rng         = quantum_random_bits(n)
    sample_pos  = [i for i, r in enumerate(rng) if r == 1][:sample_size]
    if len(sample_pos) < sample_size:
        remaining  = [i for i in range(n) if i not in sample_pos]
        sample_pos += remaining[:sample_size - len(sample_pos)]
    sample_pos = sorted(sample_pos[:sample_size])
    errors     = sum(1 for i in sample_pos if alice_key[i] != bob_key[i])
    return errors / sample_size, sample_pos


print("Protocol functions defined.")

Protocol functions defined.


In [17]:
NUM_QUBITS      = 200
QBER_THRESHOLD  = 0.11
INTERCEPT_RATE  = 1.0   # Eve intercepts 100% of qubits

# Alice
alice_bits  = quantum_random_bits(NUM_QUBITS)
alice_bases = quantum_random_bits(NUM_QUBITS)

alice_circuits = [
    alice_prepare_qubit(alice_bits[i], alice_bases[i])
    for i in range(NUM_QUBITS)
]

# Eve (Intercept-Resend)
eve_bases         = quantum_random_bits(NUM_QUBITS)  # Eve's random basis choices
intercept_flags   = quantum_random_bits(NUM_QUBITS)  # Which qubits Eve intercepts
# With INTERCEPT_RATE=1.0, Eve intercepts all; use flags for partial attack too

eve_measurements   = []   # What Eve measured (her stolen key estimate)
forwarded_circuits = []   # Qubits forwarded to Bob

for i in range(NUM_QUBITS):
    # Determine if Eve intercepts this qubit
    # For full attack, she always intercepts
    eve_intercepts = (np.random.random() < INTERCEPT_RATE)

    if eve_intercepts:
        # Eve measures in her chosen basis and re-prepares
        measured_bit, new_circuit = measure_qubit(alice_circuits[i], eve_bases[i])
        eve_measurements.append(measured_bit)
        forwarded_circuits.append(new_circuit)  # Bob receives Eve's resent qubit
    else:
        # Eve passes the qubit untouched
        eve_measurements.append(None)
        forwarded_circuits.append(alice_circuits[i])

print(f"Eve intercepted: {sum(1 for e in eve_measurements if e is not None)} / {NUM_QUBITS} qubits")

# Bob
bob_bases   = quantum_random_bits(NUM_QUBITS)
bob_results = [
    bob_measure(forwarded_circuits[i], bob_bases[i])
    for i in range(NUM_QUBITS)
]

# Sifting
alice_sifted, bob_sifted, matching = sift_key(alice_bases, bob_bases, alice_bits, bob_results)
print(f"Sifted key length: {len(alice_sifted)} bits ({100*len(alice_sifted)/NUM_QUBITS:.1f}%)")

# QBER Estimation
qber, sample_pos = estimate_qber(alice_sifted, bob_sifted)
print(f"\nQBER: {qber:.2%}  (threshold: {QBER_THRESHOLD:.0%})")

if qber > QBER_THRESHOLD:
    print(f"\nATTACK DETECTED! QBER {qber:.2%} exceeds threshold. Protocol aborted.")
    print("Alice and Bob discard the key and do not communicate further.")
else:
    print(f"\nNo attack detected (QBER ≤ threshold). Key accepted.")

Eve intercepted: 200 / 200 qubits
Sifted key length: 89 bits (44.5%)

QBER: 40.91%  (threshold: 11%)

ATTACK DETECTED! QBER 40.91% exceeds threshold. Protocol aborted.
Alice and Bob discard the key and do not communicate further.


In [18]:
INTERCEPT_RATE_PARTIAL = 0.30  # Eve intercepts only 30% of qubits
# Expected QBER = 0.30 * 0.25 = 7.5% (may sneak under threshold)

# Alice
alice_bits_p  = quantum_random_bits(NUM_QUBITS)
alice_bases_p = quantum_random_bits(NUM_QUBITS)
alice_circs_p = [alice_prepare_qubit(alice_bits_p[i], alice_bases_p[i]) for i in range(NUM_QUBITS)]

# Eve (Partial)
eve_bases_p       = quantum_random_bits(NUM_QUBITS)
forwarded_p       = []
num_intercepted   = 0

for i in range(NUM_QUBITS):
    if np.random.random() < INTERCEPT_RATE_PARTIAL:
        _, new_qc = measure_qubit(alice_circs_p[i], eve_bases_p[i])
        forwarded_p.append(new_qc)
        num_intercepted += 1
    else:
        forwarded_p.append(alice_circs_p[i])

# Bob
bob_bases_p   = quantum_random_bits(NUM_QUBITS)
bob_results_p = [bob_measure(forwarded_p[i], bob_bases_p[i]) for i in range(NUM_QUBITS)]


# Sifting & QBER
alice_sifted_p, bob_sifted_p, _ = sift_key(alice_bases_p, bob_bases_p, alice_bits_p, bob_results_p)
qber_p, _ = estimate_qber(alice_sifted_p, bob_sifted_p)

print(f"Partial intercept attack ({INTERCEPT_RATE_PARTIAL:.0%} of qubits)")
print(f"Eve intercepted: {num_intercepted} / {NUM_QUBITS}")
print(f"QBER: {qber_p:.2%}  (threshold: {QBER_THRESHOLD:.0%})")

if qber_p > QBER_THRESHOLD:
    print(f"\nATTACK DETECTED! Protocol aborted.")
else:
    print(f"\nAttack NOT detected at this sample size.")
    print("Eve may have obtained partial key information undetected.")
    print(f"This shows why the intercept rate threshold matters.")

Partial intercept attack (30% of qubits)
Eve intercepted: 67 / 200
QBER: 7.41%  (threshold: 11%)

Attack NOT detected at this sample size.
Eve may have obtained partial key information undetected.
This shows why the intercept rate threshold matters.


In [19]:
# Conceptual PNS attack — no quantum disturbance, zero QBER

MULTI_PHOTON_RATE = 0.10  # 10% of pulses are multi-photon (theoretical)

print("=== Photon Number Splitting (PNS) Attack — Conceptual ===")
print(f"Assumed multi-photon pulse rate: {MULTI_PHOTON_RATE:.0%}")
print()
print("In an ideal single-photon simulation, PNS is not possible.")
print("Eve can obtain information from at most:")
print(f"  {MULTI_PHOTON_RATE:.0%} × 50% (basis match) = {MULTI_PHOTON_RATE*0.5:.0%} of key bits")
print(f"  with ZERO induced errors.")
print()
print("Countermeasure: Decoy state protocol limits Eve's information")
print("from multi-photon pulses to a negligible fraction.")

=== Photon Number Splitting (PNS) Attack — Conceptual ===
Assumed multi-photon pulse rate: 10%

In an ideal single-photon simulation, PNS is not possible.
Eve can obtain information from at most:
  10% × 50% (basis match) = 5% of key bits
  with ZERO induced errors.

Countermeasure: Decoy state protocol limits Eve's information
from multi-photon pulses to a negligible fraction.
